In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/10/07 06:05:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
bucket = "crypto-data-lake"

In [4]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01"
df = spark.read.parquet(output_path)

25/10/04 12:45:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
df.show()

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|ingest_date|    ingest_timestamp|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------+
|  3640494121|115764.07| 0.22677|    5122977554|   5122977554|1754006400328945|          true|         true| 2025-10-04|2025-10-04 10:02:...|
|  3640494122|115764.08| 0.00145|    5122977555|   5122977555|1754006400345714|         false|         true| 2025-10-04|2025-10-04 10:02:...|
|  3640494123|115764.08|  2.1E-4|    5122977556|   5122977556|1754006400350235|         false|         true| 2025-10-04|2025-10-04 10:02:...|
|  3640494124|115764.08|  4.1E-4|    5122977557|   5122977557|1754006400492405|         false|         true| 2025-10-04|2025-10-04 10:02:...|
|  364

In [6]:
# .orderBy("timestamp", ascending=True) \
# .sample(0.001) \
# .select(["agg_trade_id", "timestamp", "timestamp_date", "timestamp_second", "group_id", "group_date"]) \
# .show(truncate=False)
df = df.withColumn("timestamp_date", F.from_unixtime(F.col("timestamp") / 1_000_000)) \
    .withColumn("timestamp_second", (F.col("timestamp") / 1_000_000).cast("long")) \
    .withColumn("group_id", (F.col("timestamp_second") / 900).cast("long")) \
    .withColumn("group_date", F.from_unixtime(F.col("group_id") * 900)) \
    .withColumn("transform_date", F.current_date()) \
    .withColumn("transform_timestamp", F.current_timestamp())

In [7]:
spark.sql("""
SHOW DATABASES
""").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [8]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS transform_db
LOCATION 's3a://crypto-data-lake/transform_zone/'
""")

DataFrame[]

In [9]:
df.writeTo("transform_db.aggtrades").tableProperty("format-version", "2").createOrReplace()

In [10]:
spark.sql("""
select count(*) from transform_db.aggtrades
""").show()

+--------+
|count(1)|
+--------+
| 1314072|
+--------+



In [11]:
spark.sql("""
select 
    group_id,
    group_date,
    max(price) as high_price,
    min(price) as low_price,
    max(agg_trade_id) as max_id,
    min(agg_trade_id) as min_id,
    sum(quantity) as volume
from 
    transform_db.aggtrades
group by group_id, group_date
""").show()

[Stage 6:>                                                          (0 + 1) / 1]

+--------+-------------------+----------+---------+----------+----------+------------------+
|group_id|         group_date|high_price|low_price|    max_id|    min_id|            volume|
+--------+-------------------+----------+---------+----------+----------+------------------+
| 1948919|2025-08-01 05:45:00| 115660.37|115385.92|3640815204|3640806867| 142.2174000000001|
| 1948901|2025-08-01 01:15:00| 115271.13|114638.65|3640653774|3640629264| 448.2353499999814|
| 1948949|2025-08-01 13:15:00| 115916.24|115525.47|3641169452|3641155320|228.84313999998787|
| 1948915|2025-08-01 04:45:00| 115699.58|115485.46|3640781833|3640775020| 132.1634600000024|
| 1948950|2025-08-01 13:30:00| 115679.82|114984.87|3641194791|3641169453| 404.7887299999752|
| 1948912|2025-08-01 04:00:00| 115648.03|115266.64|3640758431|3640746859| 199.9337699999932|
| 1948942|2025-08-01 11:30:00| 115230.76|115056.95|3641075157|3641067981| 137.7531500000022|
| 1948991|2025-08-01 23:45:00|  113400.0|113211.26|3641808192|36418025

In [12]:
spark.sql("""
select 
    agg_trade_id,
    price,
    quantity,
    timestamp
from 
    transform_db.aggtrades
""").show()

+------------+---------+--------+----------------+
|agg_trade_id|    price|quantity|       timestamp|
+------------+---------+--------+----------------+
|  3640494121|115764.07| 0.22677|1754006400328945|
|  3640494122|115764.08| 0.00145|1754006400345714|
|  3640494123|115764.08|  2.1E-4|1754006400350235|
|  3640494124|115764.08|  4.1E-4|1754006400492405|
|  3640494125|115764.08|  8.0E-5|1754006400636161|
|  3640494126|115764.07| 0.00444|1754006400669861|
|  3640494127|115764.07| 0.10877|1754006400781656|
|  3640494128|115764.06|  1.0E-4|1754006400781656|
|  3640494129|115762.89|  1.5E-4|1754006400781656|
|  3640494130|115762.88| 0.28528|1754006400781656|
|  3640494131|115762.67|  5.0E-5|1754006400781656|
|  3640494132|115761.43|   0.023|1754006400781682|
|  3640494133|115761.43|  4.0E-4|1754006400781814|
|  3640494134|115761.43|  0.0231|1754006400782004|
|  3640494135|115761.43|  3.0E-4|1754006400782028|
|  3640494136|115761.43| 0.16322|1754006400782053|
|  3640494137|115761.43| 0.1009

In [13]:
df_kline = spark.sql("""
select 
    group_id,
    group_date,
    first(timestamp, true) as open_time,
    round(first(price, true), 2) as open_price,
    round(max(price), 2) as high_price,
    round(min(price), 2) as low_price,
    round(last(price, true), 2) as close_price,
    round(sum(quantity), 2) as volume,
    last(timestamp, true) as close_time
from transform_db.aggtrades
group by group_id, group_date
order by group_id
""")

In [14]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS serving_db
LOCATION 's3a://crypto-data-lake/serving_zone/'
""")

DataFrame[]

In [15]:
df_kline.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.16|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.54|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.43|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.76|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.88|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.9| 11527

In [16]:
df_kline.writeTo("serving_db.klines").tableProperty("format-version", "2").createOrReplace()

In [4]:
spark.sql("SELECT * FROM serving_db.klines").show()

25/10/04 13:32:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.16|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.54|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.43|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.76|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.88|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.9| 11527

In [3]:
df_sorted = (
    spark.sql("SELECT * FROM serving_db.sma7")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

25/09/25 10:34:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("ema7", types.DoubleType(), True)
])

In [5]:
from decimal import Decimal, getcontext, ROUND_HALF_UP

# set precision high enough for finance data
getcontext().prec = 28  

def ema_in_chunks(iterator):  # one stream iterator per partition
    alpha = Decimal(2) / Decimal(7 + 1)  # keep alpha as Decimal
    prev = None
    for pdf in iterator:  # 10,000 rows pandas dataframe for a chunk
        ema = []
        for price in pdf["close_price"]:
            price_dec = Decimal(str(price))  # convert to Decimal exactly
            if prev is None:
                prev = Decimal(str(pdf["ma7"].iloc[0]))  # initialize with SMA7
            else:
                prev = alpha * price_dec + (Decimal(1) - alpha) * prev
            # emulate Spark's round(..., 2)
            ema.append(float(prev.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)))
        pdf["ema7"] = ema
        pdf = pdf[[*pdf.columns[:-1], "ema7"]]
        yield pdf

In [6]:
ema_df = df_sorted.mapInPandas(ema_in_chunks, schema)

In [8]:
ema_df.writeTo("serving_db.ema7").tableProperty("format-version", "2").createOrReplace()

In [8]:
!jupyter nbconvert --to script transform_job.ipynb

[NbConvertApp] Converting notebook transform_job.ipynb to script
[NbConvertApp] Writing 5519 bytes to transform_job.py
